In [ ]:
import os
import re
import sys
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend — must be before pyplot import
import matplotlib.pyplot as plt
from datasets import load_dataset
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR = "/home/valenbonas/Documents/Investigacion_doctorado/new_siglip2/Datasets/chart2code"
IMG_DIR  = os.path.join(BASE_DIR, "images")
SVG_DIR  = os.path.join(BASE_DIR, "svgs")
CODE_DIR = os.path.join(BASE_DIR, "code")

os.makedirs(IMG_DIR,  exist_ok=True)
os.makedirs(SVG_DIR,  exist_ok=True)
os.makedirs(CODE_DIR, exist_ok=True)

# ── Load dataset ───────────────────────────────────────────────────────────────
print("Loading dataset...")
dataset = load_dataset("CSU-JPG/Chart2Code", "level1_direct", split="train")
print(f"  → {len(dataset)} samples found\n")

# ── Save loop ──────────────────────────────────────────────────────────────────
skipped = 0

for sample in tqdm(dataset, desc="Processing samples"):
    sample_id  = sample["sample_id"]
    chart_type = sample["chart_type"]
    code       = sample["python_code"]
    image      = sample["input_chart_image"]  # PIL Image

    # 1) Save original PNG
    img_path = os.path.join(IMG_DIR, f"{sample_id}.png")
    image.save(img_path)

    # 2) Save Python code
    code_path = os.path.join(CODE_DIR, f"{sample_id}.py")
    with open(code_path, "w") as f:
        f.write(code)

    # 3) Run modified code to produce SVG
    svg_path = os.path.join(SVG_DIR, f"{sample_id}.svg")

    # Patch the code: replace plt.savefig(...) with SVG output path
    # Also remove any plt.show() calls
    patched_code = re.sub(
        r'plt\.savefig\([^)]*\)',
        f'plt.savefig("{svg_path}", format="svg", bbox_inches="tight")',
        code
    )
    patched_code = re.sub(r'plt\.show\(\)', '', patched_code)

    try:
        exec_globals = {"__name__": "__main__"}
        exec(compile(patched_code, sample_id, "exec"), exec_globals)
        plt.close("all")  # Free memory after each exec
    except Exception as e:
        tqdm.write(f"  ⚠️  SVG generation failed for {sample_id}: {e}")
        plt.close("all")
        skipped += 1

print(f"\nDone! Saved to {BASE_DIR}")
print(f"  ✓ {len(dataset) - skipped} SVGs generated")
print(f"  ⚠️  {skipped} SVGs skipped (exec errors)")

In [ ]:
import os
import re
import json
import shutil
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR     = "/home/valenbonas/Documents/Investigacion_doctorado/new_siglip2/Datasets/chart2coder160k"
RAW_DIR      = os.path.join(BASE_DIR, "_raw")
IMG_DIR      = os.path.join(BASE_DIR, "images")
SVG_DIR      = os.path.join(BASE_DIR, "svgs")
CODE_DIR     = os.path.join(BASE_DIR, "code")

images_extract = os.path.join(RAW_DIR, "images")
json_extract   = os.path.join(RAW_DIR, "jsons")

for d in [IMG_DIR, SVG_DIR, CODE_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Load all samples from the 3 JSON files ────────────────────────────────────
json_files = list(Path(json_extract).rglob("*.json"))
all_samples = []
for jf in json_files:
    with open(jf) as f:
        all_samples.extend(json.load(f))

print(f"Total samples: {len(all_samples)}\n")

# ── Processing loop ───────────────────────────────────────────────────────────
skipped_img  = 0
skipped_svg  = 0

for sample in tqdm(all_samples, desc="Processing samples"):
    sample_id = str(sample["id"])

    # Extract the matplotlib code from the gpt conversation turn
    code = None
    for turn in sample["conversations"]:
        if turn["from"] == "gpt":
            code = turn["value"]
            break

    if code is None:
        tqdm.write(f"  ⚠️  No code found for id {sample_id}, skipping")
        skipped_svg += 1
        continue

    # 1) Copy original PNG
    src_img = os.path.join(RAW_DIR, sample["image"])  # e.g. _raw/images/1.png
    dst_img = os.path.join(IMG_DIR, f"{sample_id}.png")
    if os.path.exists(src_img):
        shutil.copy2(src_img, dst_img)
    else:
        tqdm.write(f"  ⚠️  Image not found: {src_img}")
        skipped_img += 1

    # 2) Save Python code
    code_path = os.path.join(CODE_DIR, f"{sample_id}.py")
    with open(code_path, "w") as f:
        f.write(code)

    # 3) Generate SVG by patching and running the code
    svg_path = os.path.join(SVG_DIR, f"{sample_id}.svg")

    # Replace any plt.savefig(...) with SVG output
    # Also strip plt.show() calls
    patched = re.sub(
        r'plt\.savefig\([^)]*\)',
        f'plt.savefig("{svg_path}", format="svg", bbox_inches="tight")',
        code
    )
    patched = re.sub(r'plt\.show\(\)', '', patched)

    # If there was no savefig in the code, append one at the end
    if 'plt.savefig' not in patched:
        patched += f'\nplt.savefig("{svg_path}", format="svg", bbox_inches="tight")\n'

    try:
        exec(compile(patched, sample_id, "exec"), {"__name__": "__main__"})
        plt.close("all")
    except Exception as e:
        tqdm.write(f"  ⚠️  SVG failed for id {sample_id}: {e}")
        plt.close("all")
        skipped_svg += 1

print(f"\nDone!")
print(f"  ✓  {len(all_samples) - skipped_svg} SVGs generated")
print(f"  ⚠️  {skipped_svg} SVGs skipped")
print(f"  ⚠️  {skipped_img} images not found")

Total samples: 0



Processing samples: 0it [00:00, ?it/s]


Done!
  ✓  0 SVGs generated
  ⚠️  0 SVGs skipped
  ⚠️  0 images not found


In [1]:
import os
import json
import shutil
import vl_convert as vlc
from pathlib import Path
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
SRC_DIR  = "/home/valenbonas/Documents/Investigacion_doctorado/new_siglip2/chart-llm/docs/data/chart"
BASE_DIR = "/home/valenbonas/Documents/Investigacion_doctorado/new_siglip2/Datasets/chartllm"
IMG_DIR  = os.path.join(BASE_DIR, "images")
SVG_DIR  = os.path.join(BASE_DIR, "svgs")
CODE_DIR = os.path.join(BASE_DIR, "code")

os.makedirs(IMG_DIR,  exist_ok=True)
os.makedirs(SVG_DIR,  exist_ok=True)
os.makedirs(CODE_DIR, exist_ok=True)

json_files = sorted(Path(SRC_DIR).glob("*.vl.json"))
print(f"Found {len(json_files)} Vega-Lite specs\n")

skipped = 0
resumed = 0

for jf in tqdm(json_files, desc="Rendering specs"):
    stem = jf.stem.replace(".vl", "")   # e.g. vl_0000

    # Skip files already rendered (allows safe resume after interruption)
    if os.path.exists(os.path.join(IMG_DIR, f"{stem}.png")):
        resumed += 1
        continue

    # 1) Copy JSON code
    shutil.copy2(jf, os.path.join(CODE_DIR, f"{stem}.json"))

    with open(jf) as f:
        spec = json.load(f)

    spec_str = json.dumps(spec)

    # 2) Render PNG
    try:
        png_bytes = vlc.vegalite_to_png(spec_str, scale=2)
        with open(os.path.join(IMG_DIR, f"{stem}.png"), "wb") as f:
            f.write(png_bytes)
    except Exception as e:
        tqdm.write(f"  ⚠️  PNG failed for {stem}: {e}")
        skipped += 1
        continue

    # 3) Render SVG
    try:
        svg_str = vlc.vegalite_to_svg(spec_str)
        with open(os.path.join(SVG_DIR, f"{stem}.svg"), "w") as f:
            f.write(svg_str)
    except Exception as e:
        tqdm.write(f"  ⚠️  SVG failed for {stem}: {e}")

print(f"\nDone! Saved to {BASE_DIR}")
print(f"  ✓  {len(json_files) - skipped - resumed} new images generated")
print(f"  ⏭️  {resumed} already done (skipped)")
print(f"  ⚠️  {skipped} specs skipped (render error)")

Found 1981 Vega-Lite specs



Rendering specs:   0%|          | 6/1981 [00:00<03:15, 10.11it/s]

  ⚠️  PNG failed for vl_0005: Vega-Lite to PNG conversion failed:
SVG has an invalid size


ERROR RangeError: Invalid array length
    at we (https://cdn.jsdelivr.net/npm/vega-scale@8.1.0/+esm:7:5254)
    at https://cdn.jsdelivr.net/npm/vega-encode@5.1.0/+esm:7:9186
    at https://cdn.jsdelivr.net/npm/vega-encode@5.1.0/+esm:7:9220
    at we.transform (https://cdn.jsdelivr.net/npm/vega-encode@5.1.0/+esm:7:9604)
    at we.evaluate (https://cdn.jsdelivr.net/npm/vega-dataflow@6.1.0/+esm:7:14954)
    at we.run (https://cdn.jsdelivr.net/npm/vega-dataflow@6.1.0/+esm:7:14809)
    at _e.evaluate (https://cdn.jsdelivr.net/npm/vega-dataflow@6.1.0/+esm:7:13401)
    at eventLoopTick (ext:core/01_core.js:175:7)
    at async _e.evaluate (https://cdn.jsdelivr.net/npm/vega-view@6.1.0/+esm:7:9118)
Rendering specs:   5%|▌         | 100/1981 [00:01<00:12, 147.41it/s]

  ⚠️  PNG failed for vl_0030: Vega-Lite to PNG conversion failed:
TypeError: Cannot read properties of undefined (reading 'marktype')
    at mr.mark (https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:52213)
    at https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:52827
    at Se (https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:18636)
    at o (https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:52813)
    at Se (https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:18636)
    at mr.mark (https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:53097)
    at https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:52827
    at Se (https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:18636)
    at o (https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:52813)
    at Se (https://cdn.jsdelivr.net/npm/vega-scenegraph@5.1.0/+esm:7:18636)
  ⚠️  PNG failed for vl_0050: Vega-Lite to PNG conversion failed:
SVG has an invalid size
  ⚠️  PNG failed f

...

  ⚠️  PNG failed for vl_1936: Vega-Lite to PNG conversion failed:
SVG has an invalid size
  ⚠️  PNG failed for vl_1937: Vega-Lite to PNG conversion failed:
SVG has an invalid size


Rendering specs:  98%|█████████▊| 1949/1981 [2:13:15<02:51,  5.35s/it]

  ⚠️  PNG failed for vl_1948: Vega-Lite to PNG conversion failed:
SVG has an invalid size


Rendering specs: 100%|██████████| 1981/1981 [2:13:49<00:00,  4.05s/it]


Done! Saved to /home/valenbonas/Documents/Investigacion_doctorado/new_siglip2/Datasets/chartllm
  ✓  126 new images generated
  ⏭️  1842 already done (skipped)
  ⚠️  13 specs skipped (render error)
